# TrackNet Ball Tracking - Training Notebook

SHOT 앱 Phase 1a: 테니스 공 추적 모델 학습

## Google Drive 업로드 가이드
1. Google Drive에 `SHOT/` 폴더 생성
2. 아래 파일들 업로드:
   - `SHOT/ml/src/` → model_tracknet.py, train_tracknet.py, augmentations_ball.py, export_tracknet_onnx.py
   - `SHOT/ml/data/ball_tracknet/ball_combined.json`
   - `SHOT/ml/data/ball_tracknet/frames/` → 19,835 jpg 파일
3. 런타임 → GPU로 변경 (T4 추천)

## 1. 환경 설정

In [ ]:
!pip install -q torch torchvision onnx onnxruntime pillow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/SHOT-AI/ml/src')

In [ ]:
# Config
DATA_ROOT = '/content/drive/MyDrive/SHOT-AI/ml/data'
ANNOTATIONS = f'{DATA_ROOT}/ball_tracknet/ball_combined.json'
FRAMES_DIR = f'{DATA_ROOT}/ball_tracknet/frames'
OUTPUT_DIR = '/content/drive/MyDrive/SHOT-AI/ml/models'

EPOCHS = 100
BATCH_SIZE = 16
LR = 1e-3

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(ANNOTATIONS), f'Annotations not found: {ANNOTATIONS}'
assert os.path.exists(FRAMES_DIR), f'Frames not found: {FRAMES_DIR}'
frame_count = len([f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg')])
print(f'Config OK - {frame_count} frames found')

## 2. 데이터 로드 + 통계

In [ ]:
import json
import re
from collections import Counter, defaultdict

with open(ANNOTATIONS) as f:
    all_ann = json.load(f)

# Visibility distribution
vis_counts = Counter(a['visibility'] for a in all_ann)
print(f'Total: {len(all_ann)}')
print(f'  Visible (1):     {vis_counts.get(1, 0)}')
print(f'  Not visible (0): {vis_counts.get(0, 0)}')
print(f'  Occluded (2):    {vis_counts.get(2, 0)}')

# Video distribution
video_counts = defaultdict(int)
for a in all_ann:
    m = re.match(r'(.+?)_frame_', a['image'])
    vid = m.group(1) if m else 'unknown'
    video_counts[vid] += 1

print(f'\nVideos: {len(video_counts)}')
for vid, cnt in sorted(video_counts.items(), key=lambda x: -x[1])[:10]:
    print(f'  {vid}: {cnt} frames')
if len(video_counts) > 10:
    print(f'  ... and {len(video_counts) - 10} more')

## 3. 데이터 샘플 시각화

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

visible_ann = [a for a in all_ann if a['visibility'] > 0]
samples = random.sample(visible_ann, min(8, len(visible_ann)))

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, ann in zip(axes.flat, samples):
    img_path = os.path.join(FRAMES_DIR, ann['image'])
    if not os.path.exists(img_path):
        ax.set_title(f'MISSING: {ann["image"][:25]}', fontsize=8, color='red')
        ax.axis('off')
        continue
    img = Image.open(img_path)
    ax.imshow(img)
    w, h = img.size
    ax.plot(ann['x'] * w, ann['y'] * h, 'r+', markersize=20, markeredgewidth=2)
    ax.set_title(ann['image'][:30], fontsize=8)
    ax.axis('off')
plt.suptitle('Ball annotation samples', fontsize=14)
plt.tight_layout()
plt.show()

# Check missing files
missing = [a['image'] for a in all_ann if not os.path.exists(os.path.join(FRAMES_DIR, a['image']))]
print(f'Missing files: {len(missing)} / {len(all_ann)}')
if missing:
    print(f'Examples: {missing[:5]}')

## 4. 학습 실행

In [ ]:
!python /content/drive/MyDrive/SHOT-AI/ml/src/train_tracknet.py \
    --data {ANNOTATIONS} \
    --frames {FRAMES_DIR} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} \
    --output-dir {OUTPUT_DIR}

## 5. 검증 - 히트맵 시각화

In [ ]:
import torch
import numpy as np
from model_tracknet import TrackNet, extract_ball_position

# Load best model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TrackNet(input_channels=9, base_filters=32).to(device)
ckpt = torch.load(f'{OUTPUT_DIR}/tracknet_best.pth', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded model from epoch {ckpt["epoch"]}, val_loss={ckpt["best_val_loss"]:.6f}')

In [ ]:
# Predict on random samples
from train_tracknet import TrackNetDataset

test_dataset = TrackNetDataset(random.sample(visible_ann, min(100, len(visible_ann))),
                                FRAMES_DIR, augment=False)

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for i, ax in enumerate(axes.flat):
    if i >= len(test_dataset):
        break
    inp, target, vis = test_dataset[i]
    with torch.no_grad():
        pred = model(inp.unsqueeze(0).to(device))[0, 0].cpu().numpy()

    # Show last frame with prediction overlay
    frame = inp[6:9].permute(1, 2, 0).numpy()  # last RGB frame
    ax.imshow(frame)
    ax.imshow(pred, alpha=0.5, cmap='hot')

    # Extract predicted position
    bx, by, conf = extract_ball_position(torch.from_numpy(pred))
    if bx is not None:
        ax.plot(bx, by, 'g+', markersize=15, markeredgewidth=2)

    # Ground truth
    gt = target[0].numpy()
    gt_max = gt.max()
    if gt_max > 0:
        gy, gx = np.unravel_index(gt.argmax(), gt.shape)
        ax.plot(gx, gy, 'bx', markersize=12, markeredgewidth=2)

    ax.set_title(f'conf={conf:.2f}', fontsize=10)
    ax.axis('off')

plt.suptitle('Green+=predicted, Blue x=ground truth', fontsize=14)
plt.tight_layout()
plt.show()

## 6. 정확도 측정

In [ ]:
from train_tracknet import TrackNetDataset
from torch.utils.data import DataLoader

# Use full validation set
video_frames = defaultdict(list)
for ann in all_ann:
    m = re.match(r'(.+?)_frame_\d+$', os.path.splitext(ann['image'])[0])
    vid = m.group(1) if m else ann['image']
    video_frames[vid].append(ann)

vids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(vids)
split = max(1, int(len(vids) * 0.8))
val_ann = []
for v in vids[split:]:
    val_ann.extend(video_frames[v])

val_ds = TrackNetDataset(val_ann, FRAMES_DIR, augment=False)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

correct = 0
total = 0
errors = []

model.eval()
with torch.no_grad():
    for inp, tgt, vis in val_loader:
        pred = model(inp.to(device))
        for i in range(len(vis)):
            if vis[i] > 0:
                total += 1
                pm = pred[i, 0].cpu()
                if pm.max() > 0.5:
                    correct += 1
                    py, px = torch.where(pm == pm.max())
                    gy, gx = torch.where(tgt[i, 0] == tgt[i, 0].max())
                    if len(py) > 0 and len(gy) > 0:
                        err = ((px[0]-gx[0]).float()**2 + (py[0]-gy[0]).float()**2).sqrt().item()
                        errors.append(err)

acc = correct / max(total, 1) * 100
mean_err = np.mean(errors) if errors else float('inf')
print(f'Detection rate: {acc:.1f}% ({correct}/{total})')
print(f'Mean position error: {mean_err:.1f}px')
print(f'Target: > 85%  |  Current: {acc:.1f}%  {"PASS" if acc > 85 else "FAIL"}')

## 7. ONNX 변환

In [ ]:
!python /content/drive/MyDrive/SHOT-AI/ml/src/export_tracknet_onnx.py \
    --checkpoint {OUTPUT_DIR}/tracknet_best.pth \
    --output {OUTPUT_DIR}/ball_tracking.onnx

print(f'\nONNX model saved to: {OUTPUT_DIR}/ball_tracking.onnx')
print(f'Size: {os.path.getsize(f"{OUTPUT_DIR}/ball_tracking.onnx") / 1024 / 1024:.2f} MB')

## 8. 모델 다운로드

학습된 모델은 Google Drive에 자동 저장됩니다:
- `SHOT/ml/models/tracknet_best.pth` (PyTorch checkpoint)
- `SHOT/ml/models/ball_tracking.onnx` (Android용 ONNX)

로컬로 다운로드하려면 아래 셀을 실행하세요.

In [ ]:
from google.colab import files
files.download(f'{OUTPUT_DIR}/ball_tracking.onnx')